# Decision Tree: Information Gain

[← Back to wiki](https://ml-viz.vercel.app/wiki/decision-tree-information-gain)

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch split scorer

In [ ]:
def gini(labels):
    _, c = np.unique(labels, return_counts=True)
    p = c / c.sum()
    return 1 - (p**2).sum()

def entropy(labels):
    _, c = np.unique(labels, return_counts=True)
    p = c / c.sum()
    return -(p * np.log2(p + 1e-300)).sum()

def weighted_impurity(y, mask, fn=gini):
    n = len(y)
    return mask.sum()/n * fn(y[mask]) + (~mask).sum()/n * fn(y[~mask])

def info_gain(y, mask, fn=gini):
    return fn(y) - weighted_impurity(y, mask, fn)

# Loan-approval running example
ages   = np.array([22,25,28,30,33,36,40,45,50,60], dtype=float)
labels = np.array([0, 0, 0, 0, 1, 1, 0, 1, 1, 1])

# Full threshold scan
thresholds = (np.sort(ages)[:-1] + np.sort(ages)[1:]) / 2
gains = [(t, info_gain(labels, ages <= t)) for t in thresholds]

print(f"{'Threshold':>12} {'Gini Gain':>12}")
for t, g in gains:
    marker = " ← best" if g == max(g2 for _,g2 in gains) else ""
    print(f"{t:>12.1f} {g:>12.4f}{marker}")

## Visualize gain vs threshold

In [ ]:
ts, gs = zip(*gains)
plt.figure(figsize=(8, 4))
plt.plot(ts, gs, 'o-', color='#6366f1')
plt.axvline(ts[np.argmax(gs)], color='#f59e0b', linestyle='--', label=f'Best: {ts[np.argmax(gs)]:.1f}')
plt.xlabel('Split threshold (Age ≤ t)')
plt.ylabel('Gini information gain')
plt.title('Information gain vs threshold')
plt.legend(); plt.tight_layout(); plt.show()

## ✏️ Your turn

**Task:** Add a second binary feature `income` = [30,45,50,60,35,70,40,80,90,65] (in thousands). Find the best *single* split across *both* features.

**Extension:** Build a depth-2 decision tree by hand: find the best split at the root, then the best split for each child node.

In [ ]:
# TODO(you): add income feature and find best split across both features
# income = np.array([30,45,50,60,35,70,40,80,90,65], dtype=float)
# For each feature + threshold, compute info_gain and track the best

In [ ]:
# best_gain should be > 0.3333 (better than age alone? or not?)
# assert best_gain >= 0.3333

<details><summary>Solution</summary>

```python
income = np.array([30,45,50,60,35,70,40,80,90,65], dtype=float)
features = {'age': ages, 'income': income}
best = (0, None, None)
for name, feat in features.items():
    for t in (np.sort(feat)[:-1]+np.sort(feat)[1:])/2:
        g = info_gain(labels, feat <= t)
        if g > best[0]:
            best = (g, name, t)
print(f'Best: {best[1]} <= {best[2]:.1f}, gain={best[0]:.4f}')
# Age <= 31.5 still wins at 0.3333
```
</details>